# 02 · Fase 2 · Indicadores y modelo

**Objetivo (65 min):** pasar de "mirar datos" a **responder una pregunta** con indicadores y, si da tiempo, con un modelo.

**Ruta**

1. De la pregunta al indicador
2. Indicadores con datos reales (temporalidad, provincias, edad)
3. El efecto composición: por qué una media puede engañar
4. Brecha "explicada" y "no explicada" con un modelo (microdatos sintéticos)
5. Un modelo predictivo… y por qué quitar la variable "sexo" no lo hace justo
6. Guardar resultados y medir el tiempo (InnoSoft Reciclator)

In [ ]:
# ⚙️ CELDA DE PREPARACIÓN · ejecútala siempre la primera (Shift + Enter)
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 30)   # que se vean todas las columnas
pd.set_option("display.width", 200)

URL_BASE = "https://innosoft-igualdad-sostenibilidad.vercel.app/data/csv/"


def ruta_de(nombre_fichero):
    """Devuelve dónde está un CSV del datatón.

    1. Si está en la carpeta datos/csv del repositorio, usa esa copia (sin internet).
    2. Si no (por ejemplo, en Google Colab), usa la URL de la web del datatón.
    """
    for carpeta in [Path("datos/csv"), Path("../datos/csv")]:   # rutas RELATIVAS
        if (carpeta / nombre_fichero).exists():
            return carpeta / nombre_fichero
    return URL_BASE + nombre_fichero


def leer(nombre_fichero):
    """Lee un CSV del datatón con las opciones que funcionan para los 15 ficheros."""
    return pd.read_csv(
        ruta_de(nombre_fichero),
        sep=";",               # las columnas se separan con punto y coma
        decimal=",",           # 27,5 significa 27.5
        thousands=".",         # 1.835.875 significa 1835875
        encoding="utf-8-sig",  # quita un carácter invisible (BOM) del principio
        na_values=["."],       # el INE escribe "." cuando no hay dato
    )


print("✅ Preparación lista")

## 1. De la pregunta al indicador

Un buen análisis empieza **antes** del código:

| Paso | Ejemplo |
|---|---|
| **Pregunta** concreta | ¿La temporalidad castiga más a las mujeres jóvenes en Andalucía? |
| **Indicador** que la responde | Tasa de temporalidad por edad y sexo, y diferencia en puntos |
| **Datos** que lo permiten | DS-008 (IECA · EPA 2025) |
| **Límite** que hay que decir | Es un *stock* (personas hoy), no contratos firmados; no distingue sector TIC |

> **% vs. puntos porcentuales (pp).** Si la temporalidad pasa del 20 % al 25 %, sube **5 puntos porcentuales** o un **25 %**. No es lo mismo: di siempre cuál usas.

## 2. Indicadores con datos reales

### 2.1 · Temporalidad por edad y sexo (DS-008)

In [ ]:
temp = leer("DS-008_temporalidad_edad_sexo_andalucia_2025.csv")

# Nos quedamos con el total de sectores, sin filas de totales de edad ni "Ambos sexos"
t = temp.query("sector == 'Total' and grupo_edad != 'Total' and sexo != 'Ambos sexos'")

# pivot: de formato "largo" (una fila por combinación) a "ancho" (una columna por sexo)
tabla_temp = t.pivot(index="grupo_edad", columns="sexo", values="tasa_temporalidad")
tabla_temp["diferencia_pp"] = tabla_temp["Mujeres"] - tabla_temp["Hombres"]
tabla_temp

### 2.2 · Temporalidad por provincia (DS-015)

In [ ]:
prov = leer("DS-015_temporalidad_provincia_sexo_andalucia_2025.csv")
tabla_prov = prov.pivot(index="provincia", columns="sexo", values="tasa_temporalidad")
tabla_prov["diferencia_pp"] = tabla_prov["Mujeres"] - tabla_prov["Hombres"]
tabla_prov.sort_values("diferencia_pp", ascending=False)

👀 En casi todas las provincias la temporalidad femenina es mayor… **excepto en Almería**. ¿Por qué? Los datos no lo dicen. Hipótesis a comprobar (no a afirmar): el peso de la agricultura intensiva con mucho empleo temporal masculino. Y recuerda: **algunas provincias tienen muestras pequeñas** en la EPA.

### 2.3 · Brecha salarial por edad (DS-006, España)

In [ ]:
edad = leer("DS-006_salario_sexo_edad_nacional_2022.csv")
cols = ["salario_medio", "p10", "p25", "p50", "p75", "p90"]
edad[cols] = edad[cols].mask(edad[cols] < 0)               # trampa: negativos -> NaN
edad = edad[edad["grupo_edad"] != "Todas las edades"]

tabla_edad = edad.pivot(index="grupo_edad", columns="sexo", values="salario_medio")
tabla_edad["brecha_%"] = (tabla_edad["Hombres"] - tabla_edad["Mujeres"]) / tabla_edad["Hombres"] * 100
tabla_edad.round(1)

## 3. El efecto composición (o por qué una media puede engañar)

Mira este dato real del INE (DS-007, España 2022):

In [ ]:
contrato = leer("DS-007_salario_tipo_contrato_nacional_2022.csv")
contrato.pivot(index="tipo_contrato", columns="sexo", values="salario_medio").round(0)

🤔 **Las mujeres con contrato temporal cobran de media MÁS que las mujeres con contrato indefinido.** ¿El contrato temporal paga más?

Casi seguro que no. Lo más probable es que **las mujeres temporales y las indefinidas sean grupos distintos** (otro sector, otra edad, otra cualificación: por ejemplo, interinas en educación o sanidad). Comparar medias de grupos con **composición** diferente puede dar resultados contraintuitivos. A esto se le llama **efecto composición** (y en su versión extrema, *paradoja de Simpson*).

👉 Lección: antes de decir "X causa Y", pregúntate **quién está en cada grupo**.

### Composición en los microdatos sintéticos

Con microdatos sí podemos comparar **dentro** de cada ocupación:

In [ ]:
micro = leer("DS-013_microdatos_sinteticos_tic.csv")
micro["sexo"] = micro["sexo"].replace({"Hombre": "Hombres", "Mujer": "Mujeres"})

por_ocupacion = micro.pivot_table(index="ocupacion", columns="sexo",
                                  values="salario_hora", aggfunc="mean")
por_ocupacion["brecha_%"] = (por_ocupacion["Hombres"] - por_ocupacion["Mujeres"]) / por_ocupacion["Hombres"] * 100
por_ocupacion["%_mujeres"] = micro.groupby("ocupacion")["sexo"].apply(lambda s: (s == "Mujeres").mean() * 100)
por_ocupacion.round(1).sort_values("%_mujeres")

In [ ]:
global_h, global_m = micro.groupby("sexo")["salario_hora"].mean()[["Hombres", "Mujeres"]]
print(f"Brecha global por hora: {(global_h - global_m) / global_h:.1%}")

La brecha global es **mayor** que la de casi todas las ocupaciones por separado. ¿Cómo puede ser? Porque las mujeres se concentran en las ocupaciones **peor pagadas** (soporte, otras tareas técnicas) y están casi ausentes de las mejor pagadas (cloud, datos, dirección). Parte de la brecha es **segregación ocupacional**; otra parte ocurre **dentro** de la misma ocupación.

## 4. Brecha "explicada" y "no explicada" con un modelo

### ¿Qué es un modelo?

Una **regresión lineal** busca la recta (o el plano) que mejor resume cómo cambia una variable (el salario) cuando cambian otras (experiencia, ocupación…). Empecemos con una sola variable:

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

X = micro[["anios_experiencia"]]          # variables explicativas (siempre una tabla)
y = micro["salario_hora"]                 # lo que queremos explicar

modelo = LinearRegression().fit(X, y)
print(f"Salario/hora con 0 años de experiencia: {modelo.intercept_:.2f} €")
print(f"Cada año de experiencia suma:           {modelo.coef_[0]:.2f} €/hora")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(micro["anios_experiencia"], y, s=4, alpha=0.3, color="gray")
xs = pd.DataFrame({"anios_experiencia": np.arange(0, 46)})
ax.plot(xs, modelo.predict(xs), color="#7a5195", linewidth=3)
ax.set_xlabel("Años de experiencia"); ax.set_ylabel("Salario por hora (€)")
ax.set_title("Cada punto es una persona SINTÉTICA; la línea es el modelo")
plt.show()

### La brecha ajustada

Ahora metemos la variable **mujer** (1 si es mujer, 0 si no) junto con otras características. El coeficiente de `mujer` responde a:

> *"Entre dos personas con la misma edad, experiencia, ocupación, contrato, jornada… ¿cuánto menos cobra la mujer?"*

Truco técnico: usamos el **logaritmo** del salario para que el coeficiente se lea como un **porcentaje**. Las variables de texto (ocupación, provincia…) se convierten en columnas de 0/1 con `pd.get_dummies` (*one-hot encoding*).

In [ ]:
def brecha_ajustada(numericas, categoricas):
    """Devuelve la brecha (%) que queda tras controlar por las variables indicadas."""
    partes = [(micro["sexo"] == "Mujeres").astype(int).rename("mujer"), micro[numericas]]
    if categoricas:
        partes.append(pd.get_dummies(micro[categoricas], drop_first=True, dtype=int))
    X = pd.concat(partes, axis=1)
    modelo = LinearRegression().fit(X, np.log(micro["salario_hora"]))
    coef_mujer = modelo.coef_[list(X.columns).index("mujer")]
    return (1 - np.exp(coef_mujer)) * 100


pasos = {
    "Sin controles":                    ([], []),
    "+ edad y experiencia":             (["edad", "anios_experiencia", "antiguedad_empresa"], []),
    "+ jornada y contrato":             (["edad", "anios_experiencia", "antiguedad_empresa"], ["jornada", "tipo_contrato"]),
    "+ ocupación":                      (["edad", "anios_experiencia", "antiguedad_empresa"], ["jornada", "tipo_contrato", "ocupacion"]),
    "+ empresa, provincia, teletrabajo": (["edad", "anios_experiencia", "antiguedad_empresa"],
                                          ["jornada", "tipo_contrato", "ocupacion", "tamano_empresa", "provincia", "teletrabajo"]),
}
descomposicion = pd.Series({nombre: brecha_ajustada(*vars_) for nombre, vars_ in pasos.items()},
                           name="brecha_por_hora_%").round(1)
descomposicion

### 🧠 Cómo leer esto (y cómo NO leerlo)

- La brecha por hora baja de ~18 % a ~10 % al comparar personas **en la misma ocupación**: casi la mitad de la diferencia está en **quién ocupa qué puesto**.
- Queda un ~10 % **sin explicar** con estas variables.

⚠️ **"No explicada" ≠ "discriminación demostrada".** Puede haber variables que no tenemos (formación, horas extra, negociación…). Y al revés: la **segregación ocupacional** también puede ser fruto de barreras. El modelo **reparte** la diferencia, no dice de quién es la culpa.

⚠️ **Datos sintéticos:** este resultado sirve para aprender el **método**. No lo presentes como un hecho del mercado laboral andaluz.

## 5. Un modelo predictivo… y una pregunta de justicia

Ahora queremos **predecir** el salario de una persona a partir de sus características. Para saber si el modelo funciona, lo evaluamos con datos que **no ha visto**: separamos *entrenamiento* (80 %) y *prueba* (20 %).

Decisión: **no le damos la variable sexo** al modelo. ¿Basta eso para que sea justo?

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

variables_num = ["edad", "anios_experiencia", "antiguedad_empresa", "horas_semana"]
variables_cat = ["ocupacion", "tipo_contrato", "tamano_empresa", "provincia", "teletrabajo"]  # ¡sin sexo!

X = pd.concat([micro[variables_num], pd.get_dummies(micro[variables_cat], drop_first=True, dtype=int)], axis=1)
y = micro["salario_bruto_anual"]

X_ent, X_prueba, y_ent, y_prueba, sexo_ent, sexo_prueba = train_test_split(
    X, y, micro["sexo"], test_size=0.2, random_state=42)    # random_state -> resultado reproducible

modelo = LinearRegression().fit(X_ent, y_ent)
pred = modelo.predict(X_prueba)

print(f"Error absoluto medio (MAE): {mean_absolute_error(y_prueba, pred):,.0f} €")
print(f"R² (0 = no explica nada, 1 = perfecto): {r2_score(y_prueba, pred):.2f}")

In [ ]:
# ¿Se equivoca igual con hombres y con mujeres?
errores = pd.DataFrame({"sexo": sexo_prueba.values, "real": y_prueba.values, "predicho": pred})
errores["error"] = errores["predicho"] - errores["real"]       # >0: el modelo predice MÁS de lo que cobra
errores.groupby("sexo")["error"].agg(["mean", "count"]).round(0)

### 🧠 Qué ha pasado

El modelo **no conoce el sexo**, pero predice a las mujeres **más** de lo que cobran y a los hombres **menos**. Es decir: con el mismo perfil, las mujeres sintéticas cobran menos de lo que "les tocaría". Quitar la variable sensible **no elimina** la desigualdad que está en los datos.

Si alguien usara un modelo así para **fijar** salarios, reproduciría (o escondería) la brecha. Por eso, en IA responsable, además del error global se mide **el error por grupo**.

## 6. Guardar resultados y medir el tiempo

### Exportar un CSV de resultados (entregable)

In [ ]:
Path("resultados").mkdir(exist_ok=True)

resultados = pd.DataFrame({
    "indicador": ["Brecha por hora sin controles", "Brecha por hora tras controlar ocupación y más"],
    "valor_pct": [descomposicion.iloc[0], descomposicion.iloc[-1]],
    "unidad": ["% (1 - exp(coef))", "% (1 - exp(coef))"],
    "fuente": ["DS-013 (SINTÉTICO)", "DS-013 (SINTÉTICO)"],
})
resultados.to_csv("resultados/resultados_ejemplo.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
resultados

### ♻️ InnoSoft Reciclator: mismo resultado, menos cómputo

El bonus del +10 % premia el código eficiente. La regla más importante en pandas: **evita los bucles fila a fila**; usa operaciones sobre columnas enteras (**vectorizadas**).

In [ ]:
import time

# ❌ Lento: un bucle que recorre las 3.000 filas una a una
inicio = time.perf_counter()
salario_mensual = []
for _, fila in micro.iterrows():
    salario_mensual.append(fila["salario_bruto_anual"] / 12)
t_bucle = time.perf_counter() - inicio

# ✅ Rápido: una operación sobre la columna entera
inicio = time.perf_counter()
salario_mensual_v = micro["salario_bruto_anual"] / 12
t_vector = time.perf_counter() - inicio

print(f"Bucle:      {t_bucle * 1000:8.2f} ms")
print(f"Vectorizado:{t_vector * 1000:8.2f} ms   ->  ~{t_bucle / t_vector:,.0f} veces más rápido")

Otras reglas del Reciclator:

- **Lee cada fichero una sola vez** y guárdalo en una variable (no lo vuelvas a descargar en cada celda).
- **Filtra pronto**: si solo usas Andalucía, quita el resto antes de agrupar.
- **No dibujes 50 gráficos** para quedarte con 4.
- Para medir una celda entera en Jupyter: escribe `%%time` en su primera línea.

## ✅ Checklist de la Fase 2

- [ ] Tengo **una pregunta** y **un indicador** que la responde.
- [ ] He calculado la brecha con **dos definiciones** (o explico por qué uso una).
- [ ] He comparado **media y mediana**.
- [ ] He pensado en la **composición** (edad, ocupación, jornada).
- [ ] Si uso DS-013, lo marco como **SINTÉTICO** en cada resultado.
- [ ] He escrito qué **no** puedo concluir.
- [ ] He guardado un **CSV de resultados** con unidades.

In [ ]:
# Tu turno 👇